# Capstone Function 4
Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

 Input | Output | Goal |
|-------|--------|------|
| 3D Array (30, 4) | 1D Array (30, ) | Maximise |

## Initial Submission

Bayesian Optimization for 4D warehouse placement hyperparameter tuning using BoTorch with GP surrogate and Expected Improvement.

### Step 1: Import Required Libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("Libraries imported!")

### Step 2: Load Data

In [ ]:
X_init = np.load('../../data/f4/updated_inputs - Week 4.npy')
y_init = np.load('../../data/f4/updated_outputs - Week 4.npy')
print(f"Loaded {X_init.shape[0]} samples, {X_init.shape[1]}D inputs")
print(f"Best value: {y_init.max():.6f} at {X_init[y_init.argmax()]}")

In [ ]:
# Visualize pair plot
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
for idx, (i, j) in enumerate(pairs):
    axes[idx].scatter(X_init[:, i], X_init[:, j], c=y_init, cmap='viridis', edgecolors='black')
    axes[idx].set_xlabel(f'Param {i+1}')
    axes[idx].set_ylabel(f'Param {j+1}')
plt.suptitle('4D Warehouse Hyperparameters - Pairwise Projections', fontweight='bold')
plt.tight_layout()
plt.show()

### Step 3: Hyperparameters

**4D Warehouse (30 samples, local optima):**
- GP Kernel: Matern 5/2 (handles complex interactions)
- Acquisition: Expected Improvement
- Restarts: 20, Raw samples: 2048 (thorough 4D search)

In [ ]:
# All inputs must be in range [0, 0.999999] per submission requirements
N_DIM = X_init.shape[1]
BOUNDS = torch.tensor([[0.0] * N_DIM, [0.999999] * N_DIM], dtype=torch.float64)
NUM_RESTARTS, RAW_SAMPLES = 20, 2048
print(f"Bounds: [0, 0.999999] for all {N_DIM}D, Restarts: {NUM_RESTARTS}, Raw samples: {RAW_SAMPLES}")

### Step 4: Train GP Model

In [ ]:
X_train = torch.tensor(X_init, dtype=torch.float64)
y_train = torch.tensor(y_init, dtype=torch.float64).unsqueeze(-1)
gp_model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)
# Check kernel structure for lengthscale access
if hasattr(gp_model.covar_module, 'base_kernel'):
    lengthscales = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
else:
    lengthscales = gp_model.covar_module.lengthscale.detach().numpy()[0]
print(f"✓ Trained! Lengthscales: {lengthscales}")

### Step 5: Optimize & Propose Next Point

In [ ]:
EI = ExpectedImprovement(gp_model, best_f=y_train.max().item())
candidate, acq_value = optimize_acqf(EI, bounds=BOUNDS, q=1, num_restarts=NUM_RESTARTS, raw_samples=RAW_SAMPLES)
next_point = candidate.detach().numpy()[0]
print(f"Next point: {next_point}")

### Step 6: Visualize Progress & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Lengthscales (feature importance)
if hasattr(gp_model.covar_module, 'base_kernel'):
    lengthscales = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
else:
    lengthscales = gp_model.covar_module.lengthscale.detach().numpy()[0]
ax1.bar(range(1, 5), lengthscales, color='steelblue', edgecolor='black')
ax1.set_xlabel('Hyperparameter')
ax1.set_ylabel('Lengthscale')
ax1.set_title('GP Lengthscales: Feature Sensitivity', fontweight='bold')
ax1.set_xticks(range(1, 5))
# Progress
best_so_far = np.maximum.accumulate(y_init)
ax2.plot(range(1, len(best_so_far)+1), best_so_far, 'b-o', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Best Value')
ax2.set_title('Optimization Progress', fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Submission ready: {next_point}")

### Visualize Expected Improvement

For higher-dimensional spaces, we visualize 1D slices of the acquisition function.
Each plot shows how EI changes along one dimension while others are fixed at the proposed point.

In [ ]:
# 1D marginal plots of Expected Improvement
n_points = 100
n_dims = len(next_point)

fig, axes = plt.subplots(1, n_dims, figsize=(4*n_dims, 4))
if n_dims == 1:
    axes = [axes]

for dim in range(n_dims):
    # Create points varying along this dimension
    X_marginal = np.tile(next_point, (n_points, 1))
    X_marginal[:, dim] = np.linspace(0, 0.999999, n_points)
    X_marginal_torch = torch.tensor(X_marginal, dtype=torch.float64)
    
    # Compute EI at each point
    with torch.no_grad():
        ei_values = EI(X_marginal_torch.unsqueeze(1)).numpy()
    
    # Plot
    axes[dim].plot(X_marginal[:, dim], ei_values, 'b-', linewidth=2)
    axes[dim].axvline(next_point[dim], color='r', linestyle='--', linewidth=2, label='Proposed')
    axes[dim].set_xlabel(f'x{dim+1}', fontsize=12)
    axes[dim].set_ylabel('Expected Improvement' if dim == 0 else '', fontsize=12)
    axes[dim].set_title(f'EI along dim {dim+1}', fontsize=11, fontweight='bold')
    axes[dim].grid(True, alpha=0.3)
    if dim == 0:
        axes[dim].legend()

plt.suptitle('Expected Improvement - 1D Marginals', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Red dashed lines show the proposed next point coordinates.")
print(f"EI is maximized when considering all dimensions jointly.")

### Step 7: Format Next Query for Submission

Format the proposed next sample point in the required submission format:
- Format: `x1-x2-x3-...-xn` where each xᵢ begins with 0
- Precision: 6 decimal places per coordinate
- Range: All values clamped to [0, 0.999999]

In [ ]:
# Format the next query for submission
def format_query(point):
    """Format point as x1-x2-...-xn with 6 decimal places, clamped to [0, 0.999999]."""
    clamped = [max(0.0, min(0.999999, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

# Clamp next_point to valid range
next_point_clamped = np.array([max(0.0, min(0.999999, x)) for x in next_point])

# Display the formatted submission query
submission_query = format_query(next_point)
print("=" * 60)
print("SUBMISSION QUERY FOR FUNCTION 4")
print("=" * 60)
print(f"\n{submission_query}\n")
print("=" * 60)
print(f"\nCoordinates breakdown:")
for i, x in enumerate(next_point, 1):
    print(f"  x{i} = {x:.6f}")
print(f"\nEI value: {acq_value.item():.6f}")
if acq_value.item() > 0.1:
    print("  -> High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  -> Moderate EI: Some exploration potential remains")
else:
    print("  -> Low EI: Approaching convergence")
if acq_value.item() > 0.1:
    print("  → High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  → Moderate EI: Some exploration potential remains")
else:
    print("  → Low EI: Approaching convergence")
print(f"Current best observed: {y_train.max().item():.6f}")

## Week 5 — Gradient Boosted Trees Surrogate

This section replaces the Gaussian Process surrogate with a **Gradient Boosted Trees (GBT)** ensemble for the 4D optimization problem (f4).

**Why Gradient Boosted Trees for f4?**
- Sequential boosting builds trees that correct residual errors from previous trees — effective for learning complex, non-linear mappings
- Handles higher-dimensional spaces (4D) better than GP which scales poorly with dimensionality
- Uncertainty estimated via an ensemble of 10 independently trained GBT models with different random seeds — standard deviation across ensemble predictions provides a diversity-based uncertainty measure
- No kernel/covariance assumptions — purely data-driven

**Acquisition Strategy:** Upper Confidence Bound (UCB) using ensemble standard deviation as uncertainty.

### Step 1: Load Week 5 Data

Load the cumulative Week 5 data (35 total samples = initial + weekly submissions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

# Load Week 5 cumulative data
X_w5 = np.load('../../data/f4/updated_inputs - Week 5.npy')
y_w5 = np.load('../../data/f4/updated_outputs - Week 5.npy')

print(f"Week 5 Data: {X_w5.shape[0]} samples, {X_w5.shape[1]} dimensions")
print(f"Input range:  [{X_w5.min():.6f}, {X_w5.max():.6f}]")
print(f"Output range: [{y_w5.min():.6f}, {y_w5.max():.6f}]")
print(f"Best observed value: {y_w5.max():.6f} at index {y_w5.argmax()}")
print(f"Best observed point: {X_w5[y_w5.argmax()]}")

### Step 2: GBT Ensemble Hyperparameters

**Hyperparameter Choices and Justifications:**

1. **n_estimators = 200**: Number of boosting rounds per model. Sufficient to capture the objective landscape without excessive overfitting on 35 samples.

2. **max_depth = 4**: Maximum tree depth per boosting round. Shallow trees (stumps to depth-4) are standard for boosting — deeper trees risk overfitting on small datasets.

3. **learning_rate = 0.1**: Step size shrinkage. Lower rates require more trees but generalize better. Paired with n_estimators to balance speed and accuracy.

4. **subsample = 0.8**: Fraction of samples used per boosting round. Stochastic gradient boosting adds regularization and introduces diversity across ensemble members.

5. **n_ensemble = 10**: Number of independently trained GBT models. Each model uses a different random seed, producing diverse predictions. Ensemble std provides uncertainty.

6. **UCB kappa = 2.0**: Exploration parameter. Slightly standard for 4D space to encourage exploration.

7. **n_candidates = 20,000**: Random candidate points for UCB evaluation.

In [ ]:
# --- GBT Ensemble Hyperparameters ---
N_ESTIMATORS = 200
MAX_DEPTH = 4
LEARNING_RATE = 0.1
SUBSAMPLE = 0.8
N_ENSEMBLE = 10
KAPPA = 2.0
N_CANDIDATES = 20000

print("GBT Ensemble Surrogate Hyperparameters:")
print(f"  n_estimators:  {N_ESTIMATORS}")
print(f"  max_depth:     {MAX_DEPTH}")
print(f"  learning_rate: {LEARNING_RATE}")
print(f"  subsample:     {SUBSAMPLE}")
print(f"  n_ensemble:    {N_ENSEMBLE}")
print(f"  UCB kappa:     {KAPPA}")
print(f"  UCB candidates: {N_CANDIDATES}")

### Step 3: Train GBT Ensemble

Train 10 independent GBT models with different random seeds. Display training scores and feature importance from the first model.

In [ ]:
# Train ensemble of GBT models
ensemble_models = []
for i in range(N_ENSEMBLE):
    model = GradientBoostingRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        subsample=SUBSAMPLE,
        random_state=42 + i
    )
    model.fit(X_w5, y_w5)
    ensemble_models.append(model)
    r2 = model.score(X_w5, y_w5)
    print(f"  Model {i+1}: Training R² = {r2:.6f}")

print()
print("GBT Ensemble Training Complete:")
print(f"  {N_ENSEMBLE} models trained")

# Feature importance from first model (representative)
print()
print("Feature Importance (Model 1):")
for i, imp in enumerate(ensemble_models[0].feature_importances_):
    print(f"  x{i+1}: {imp:.4f} ({'*' * int(imp * 20)})")

### Step 4: UCB Acquisition Function

Compute the Upper Confidence Bound using:
- **mu(x)** = mean prediction across the 10-model ensemble
- **sigma(x)** = standard deviation across the 10 ensemble predictions
- **UCB(x) = mu(x) + kappa * sigma(x)** where kappa = 2.0

In [ ]:
# Generate random candidate points
np.random.seed(42)
candidates = np.random.uniform(0, 0.999999, size=(N_CANDIDATES, 4))

# Get predictions from all ensemble members
ensemble_preds = np.array([m.predict(candidates) for m in ensemble_models])

# Ensemble mean and std
mu = ensemble_preds.mean(axis=0)
sigma = ensemble_preds.std(axis=0)

# UCB acquisition
ucb = mu + KAPPA * sigma

# Select best candidate
best_idx = np.argmax(ucb)
next_point_w5 = np.clip(candidates[best_idx], 0.0, 0.999999)

print("UCB Acquisition Results:")
print(f"  Best UCB value:      {ucb[best_idx]:.6f}")
print(f"  Ensemble mean:       {mu[best_idx]:.6f}")
print(f"  Ensemble std:        {sigma[best_idx]:.6f}")
print(f"  Next sample point:   {next_point_w5}")
print()
print("Ensemble prediction statistics:")
print(f"  Mean of mu:    {mu.mean():.6f}")
print(f"  Max of mu:     {mu.max():.6f}")
print(f"  Mean of sigma: {sigma.mean():.6f}")
print(f"  Max of sigma:  {sigma.max():.6f}")

### Step 5: Visualize GBT Surrogate

2D slice plots showing ensemble mean and uncertainty. For 4D data, we fix the two least important dimensions at their best observed values and plot the two most important.

In [ ]:
# Identify top 2 dimensions by feature importance
importances = ensemble_models[0].feature_importances_
top2 = np.argsort(importances)[-2:]  # indices of top 2
fixed_dims = [d for d in range(4) if d not in top2]
best_point = X_w5[y_w5.argmax()]

print(f"Top 2 dimensions: x{top2[0]+1} (imp={importances[top2[0]]:.4f}), x{top2[1]+1} (imp={importances[top2[1]]:.4f})")
print(f"Fixed dimensions: " + ", ".join([f"x{d+1}={best_point[d]:.6f}" for d in fixed_dims]))

# Create 2D grid for top 2 dimensions
n_grid = 50
d0_grid = np.linspace(0, 1, n_grid)
d1_grid = np.linspace(0, 1, n_grid)
D0, D1 = np.meshgrid(d0_grid, d1_grid)

# Build full grid points with fixed dimensions
grid_points = np.zeros((n_grid * n_grid, 4))
grid_points[:, top2[0]] = D0.ravel()
grid_points[:, top2[1]] = D1.ravel()
for d in fixed_dims:
    grid_points[:, d] = best_point[d]

# Ensemble predictions on grid
grid_ensemble_preds = np.array([m.predict(grid_points) for m in ensemble_models])
grid_mu = grid_ensemble_preds.mean(axis=0).reshape(n_grid, n_grid)
grid_sigma = grid_ensemble_preds.std(axis=0).reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Ensemble mean
ax1 = axes[0]
c1 = ax1.contourf(D0, D1, grid_mu, levels=30, cmap='viridis')
ax1.scatter(X_w5[:, top2[0]], X_w5[:, top2[1]], c='red', edgecolors='white', s=60, zorder=5, label='Observed')
ax1.scatter(next_point_w5[top2[0]], next_point_w5[top2[1]], c='yellow', marker='*', s=200, edgecolors='black', zorder=6, label='Next point')
fixed_str = ", ".join([f"x{d+1}={best_point[d]:.2f}" for d in fixed_dims])
ax1.set_xlabel(f'x{top2[0]+1}'); ax1.set_ylabel(f'x{top2[1]+1}')
ax1.set_title(f'GBT Ensemble Mean ({fixed_str})')
ax1.legend(loc='lower right', fontsize=8)
plt.colorbar(c1, ax=ax1)

# Plot 2: Ensemble uncertainty
ax2 = axes[1]
c2 = ax2.contourf(D0, D1, grid_sigma, levels=30, cmap='YlOrRd')
ax2.scatter(X_w5[:, top2[0]], X_w5[:, top2[1]], c='blue', edgecolors='white', s=60, zorder=5, label='Observed')
ax2.set_xlabel(f'x{top2[0]+1}'); ax2.set_ylabel(f'x{top2[1]+1}')
ax2.set_title(f'Ensemble Std ({fixed_str})')
ax2.legend(loc='lower right', fontsize=8)
plt.colorbar(c2, ax=ax2)

# Plot 3: Feature importance
ax3 = axes[2]
dims = [f'x{i+1}' for i in range(4)]
colors = ['steelblue' if i in top2 else 'lightgray' for i in range(4)]
ax3.barh(dims, importances, color=colors)
ax3.set_xlabel('Importance')
ax3.set_title('Feature Importance (GBT)')
ax3.set_xlim(0, 1)

plt.tight_layout()
plt.show()

### Step 6: Convergence Plot

Running maximum (best observed value) across all observations.

In [ ]:
# Convergence plot
running_max = np.maximum.accumulate(y_w5)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(y_w5) + 1), running_max, 'b-o', markersize=6, label='Running Maximum')
plt.scatter(range(1, len(y_w5) + 1), y_w5, c='gray', alpha=0.5, s=30, label='Individual Observations')
plt.axvline(x=15.5, color='red', linestyle='--', alpha=0.5, label='Initial -> Weekly boundary')
plt.xlabel('Observation Number')
plt.ylabel('Objective Value')
plt.title('Function 4 - Convergence Plot (Week 5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best observed value: {y_w5.max():.6f}")
print(f"Achieved at observation: {y_w5.argmax() + 1}")

### Step 7: Format Submission Query

Format the proposed next sample point as `x1-x2-x3-x4` with 6 decimal places.

In [ ]:
# Format submission query
def format_query(point):
    clamped = [max(0.0, min(0.999999, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

submission_query_w5 = format_query(next_point_w5)

print("=" * 60)
print("WEEK 5 SUBMISSION QUERY FOR FUNCTION 4")
print("=" * 60)
print(f"Surrogate: GBT Ensemble ({N_ENSEMBLE} models x {N_ESTIMATORS} trees)")
print(f"Acquisition: UCB (kappa={KAPPA})")
print(f"Next point: {next_point_w5}")
print(f"Ensemble mean: {mu[best_idx]:.6f}")
print(f"Ensemble std:  {sigma[best_idx]:.6f}")
print(f"")
print(f">>> SUBMISSION: {submission_query_w5}")
print("=" * 60)

### Model Comparison

**Gradient Boosted Trees vs GP (Initial Section):**
- The GP model uses a Matern 5/2 kernel with Expected Improvement and handles 4D via ARD lengthscales that learn dimension relevance.
- GBT builds an ensemble of boosted decision trees — each tree corrects residual errors, learning a progressively refined approximation.
- Uncertainty in GP comes from the posterior distribution (well-calibrated). GBT uncertainty comes from training 10 independent models with different random seeds — less principled but practical.
- For f4's 4D problem with 35 samples, GBT can capture non-linear interactions without the cubic scaling issues of GP's covariance matrix inversion.
- Key trade-off: GP provides calibrated uncertainty while GBT provides more flexible function approximation at the cost of less principled uncertainty estimates.